# 🛡️ Cybersecurity & Data Breach Intelligence System
### Global Breach Analysis, Machine Learning Severity Prediction & Threat Intelligence Architecture

**Author:** Cybersecurity Data Intelligence Engineering Team  
**Dataset:** Kaggle - List of Biggest Data Breaches & Hacks  
**Technologies:** Python 3, Pandas, NumPy, Matplotlib, Seaborn, Scikit-Learn, XGBoost, MySQL Server, Tableau, JupyterLab  

---

### 📋 Project Overview & Architecture
Data breaches represent an existential threat to modern digital economies, resulting in billions of compromised credentials, severe regulatory penalties, and reputational damage. This comprehensive data science project provides an end-to-end framework to:
1. **Collect & Ingest:** Multi-decade global data breach incident logs.
2. **Clean & Standardize:** Normalize irregular records, clean missing data, and create historical decade categorizations.
3. **Engineer Cyber Features:** Formulate log-transformed metrics, entity classification (Major Tech vs Other), and standardized attack categories.
4. **Conduct Exploratory Data Analysis:** Render 10+ professional cybersecurity intelligence visualizations.
5. **Compute Cyber Statistics:** Synthesize global impact metrics, year-over-year expansion rates, and vector risk weights.
6. **Deploy Machine Learning Models:** Train, tune, and evaluate **Random Forest**, **XGBoost**, and **Logistic Regression** to predict breach severity tiers.
7. **Generate Threat Intelligence:** Deliver actionable threat intelligence dossiers for CISOs, security researchers, and enterprise stakeholders.
8. **Integrate Relational Database:** Store the warehouse dataset in **MySQL (`cybersecurity_db`)** and execute 5 advanced enterprise analytical SQL queries.
9. **Export & Tableau Enablement:** Persist structured feeds ready for interactive dashboard visualization.


In [1]:
# ==========================================
# IMPORT CORE DATA SCIENCE & ML LIBRARIES
# ==========================================

# Standard system and file management modules
import os
import sys
import warnings

# Suppress minor depreciation warnings to maintain clean presentation
warnings.filterwarnings('ignore')

# Numerical computing and matrix manipulation
import numpy as np

# Tabular data structure and manipulation
import pandas as pd

# Statistical data visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Scikit-Learn preprocessing, modeling, and evaluation metrics
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

# Gradient Boosted Trees for high-performance classification
from xgboost import XGBClassifier

# MySQL Connector for database integration
sys.path.append(r'C:\Users\HP\AppData\Roaming\Python\Python313\site-packages')
import mysql.connector

# Configure aesthetic visual standards for all plots
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
plt.rcParams['figure.dpi'] = 120

print(" All core libraries successfully imported and visual parameters initialized.")


---
## STEP 1 — DATA COLLECTION AND LOADING
In this phase, we ingest the global data breaches dataset into a Pandas DataFrame. We perform an initial diagnostic audit by inspecting dataframe dimensions, column definitions, data types, null distributions, and distinct categorical values.


In [2]:
# ==========================================
# STEP 1: DATA COLLECTION AND LOADING
# ==========================================

try:
    # Define primary dataset path
    dataset_path = 'data_breaches.csv'
    
    # Check if primary path exists, fallback if necessary
    if not os.path.exists(dataset_path):
        dataset_path = r'C:\Users\HP\Cybersecurity\data_breaches.csv'
        
    # Load dataset into pandas dataframe
    df = pd.read_csv(dataset_path)
    print(f" Successfully loaded dataset from: {dataset_path}")
    
    # 1. Print Shape of dataset
    print(f"\n--- 1. Dataset Dimensions ---")
    print(f"Total Rows (Incidents): {df.shape[0]}")
    print(f"Total Columns (Attributes): {df.shape[1]}")
    
    # 2. Print Column Names
    print(f"\n--- 2. Column Attributes ---")
    print(df.columns.tolist())
    
    # 3. Print Data Types
    print(f"\n--- 3. Attribute Data Types ---")
    print(df.dtypes)
    
    # 4. Print First 5 Rows (Head)
    print(f"\n--- 4. Preview First 5 Records ---")
    print(df.head())
    
    # 5. Check for Missing Values
    print(f"\n--- 5. Missing Values Audit ---")
    missing_audit = pd.DataFrame({
        'Missing Values': df.isnull().sum(),
        'Percentage (%)': (df.isnull().sum() / len(df) * 100).round(2)
    })
    print(missing_audit)
    
    # 6. Unique Values for Key Categorical Columns
    print(f"\n--- 6. Unique Values for Key Categorical Columns ---")
    print(f"Unique Methods Count: {df['Method'].nunique()}")
    print("Sample Methods:", df['Method'].dropna().unique()[:10])
    
    print(f"\nUnique Sectors Count: {df['Sector'].nunique()}")
    print("Sample Sectors:", df['Sector'].dropna().unique()[:10])
    
    print(f"\nUnique Organisations Count: {df['Organisation'].nunique()}")
    print("Sample Organisations:", df['Organisation'].dropna().unique()[:10])

except Exception as err:
    print(f" Error encountered in Step 1: {err}")


---
## STEP 2 — DATA CLEANING & TRANSFORMATION
Real-world cybersecurity incident reports frequently suffer from noisy data: string-formatted record counts with commas or abbreviations (e.g. `'3M'`, `'100,000'`), unstandardized casing, and non-integer year markers. 

In this step, we:
- Clean and convert **Records Lost** to standard integer values.
- Clean **Year** and handle non-numeric values.
- Standardize all textual columns to **Title Case**.
- Formulate a 4-tier **Severity** index based on records breached:
  - **Critical:** $> 100,000,000$ records
  - **High:** $10,000,000$ to $100,000,000$ records
  - **Medium:** $1,000,000$ to $10,000,000$ records
  - **Low:** $< 1,000,000$ records
- Derive a historical **Decade** grouping (`1980s`, `1990s`, `2000s`, `2010s`, `2020s`).


In [3]:
# ==========================================
# STEP 2: DATA CLEANING & TRANSFORMATION
# ==========================================

try:
    # Create working copy to prevent mutating raw data
    df_cleaned = df.copy()

    # Function to parse and standardize Records Lost into pure numeric float
    def parse_records_lost(val):
        # Handle null values
        if pd.isna(val):
            return np.nan
        # Convert string to uppercase stripped text
        s = str(val).strip()
        # Handle edge cases like 'ALL' (e.g., Animoto credential leak)
        if s.upper() == 'ALL':
            return 25000000.0
        # Handle 'M' suffix (Millions)
        if s.upper().endswith('M'):
            try:
                return float(s[:-1].replace(',', '').strip()) * 1e6
            except:
                return np.nan
        # Handle 'K' suffix (Thousands)
        if s.upper().endswith('K'):
            try:
                return float(s[:-1].replace(',', '').strip()) * 1e3
            except:
                return np.nan
        # Remove commas, quotes, and non-numeric artifacts
        cleaned_chars = ''.join(c for c in s if c.isdigit() or c == '.')
        try:
            return float(cleaned_chars) if cleaned_chars else np.nan
        except:
            return np.nan

    # Apply numeric parsing to Records Lost
    df_cleaned['Records Lost'] = df_cleaned['Records Lost'].apply(parse_records_lost)

    # Impute missing records with sector median or global median
    median_records = df_cleaned['Records Lost'].median()
    df_cleaned['Records Lost'] = df_cleaned['Records Lost'].fillna(median_records).astype('int64')

    # Ensure Year is integer
    df_cleaned['Year'] = pd.to_numeric(df_cleaned['Year'], errors='coerce')
    median_year = int(df_cleaned['Year'].median())
    df_cleaned['Year'] = df_cleaned['Year'].fillna(median_year).astype(int)

    # Standardize string columns to clean Title Case
    text_attributes = ['Organisation', 'Sector', 'Method']
    for col in text_attributes:
        df_cleaned[col] = df_cleaned[col].fillna('Unknown').astype(str).str.strip().str.title()

    # Define Severity tiered categorization function
    def calculate_severity(records):
        if records > 100000000:
            return 'Critical'
        elif records >= 10000000:
            return 'High'
        elif records >= 1000000:
            return 'Medium'
        else:
            return 'Low'

    # Apply Severity mapping
    df_cleaned['Severity'] = df_cleaned['Records Lost'].apply(calculate_severity)

    # Map Year to Decade bins
    def derive_decade(year):
        if 1980 <= year < 1990:
            return '1980s'
        elif 1990 <= year < 2000:
            return '1990s'
        elif 2000 <= year < 2010:
            return '2000s'
        elif 2010 <= year < 2020:
            return '2010s'
        elif year >= 2020:
            return '2020s'
        else:
            return 'Other'

    # Apply Decade derivation
    df_cleaned['Decade'] = df_cleaned['Year'].apply(derive_decade)

    # Display Cleaning Summary
    print("="*60)
    print("DATA CLEANING COMPLETED SUCCESSFULLY")
    print("="*60)
    print(f"Total Records Cleaned: {len(df_cleaned)}")
    print(f"Remaining Missing Values: {df_cleaned.isnull().sum().sum()}")
    print("\nSeverity Tier Breakdown:")
    print(df_cleaned['Severity'].value_counts())
    print("\nDecade Breakdown:")
    print(df_cleaned['Decade'].value_counts()[['1980s', '1990s', '2000s', '2010s', '2020s']])

except Exception as err:
    print(f" Error in Step 2: {err}")


---
## STEP 3 — FEATURE ENGINEERING
To empower downstream machine learning and quantitative modeling, we formulate specialized domain features:
1. **`Log_Records`:** $\log_{10}(\text{Records Lost})$ to stabilize high skewness and extreme mega-breach outliers.
2. **`Is_Recent`:** Binary indicator ($1$ if incident $\ge 2015$, else $0$).
3. **`Is_Major_Tech`:** Binary flag for top 10 tech titans: *Yahoo, Facebook, LinkedIn, Twitter, Google, Microsoft, Amazon, Apple, Adobe, eBay*.
4. **`Attack_Category`:** Harmonized taxonomy grouping the messy attack methods into 6 actionable categories:
   - **Hacked:** External unauthorized system intrusion / exploitation
   - **Accidentally Published:** Unsecured open storage, accidental database exposure
   - **Poor Security:** Misconfiguration, default credentials, weak hashing
   - **Inside Job:** Rogue employee, insider theft, credential misuse
   - **Lost Device:** Physical laptop, backup tape, or flash drive theft
   - **Other:** Unspecified or composite threat vectors


In [4]:
# ==========================================
# STEP 3: FEATURE ENGINEERING
# ==========================================

try:
    df_features = df_cleaned.copy()

    # 1. Log_Records: Log10 of Records Lost to handle extreme outliers
    # Use max(records, 1) to avoid math domain errors
    df_features['Log_Records'] = np.log10(np.maximum(df_features['Records Lost'], 1.0))

    # 2. Is_Recent: 1 if Year >= 2015 else 0
    df_features['Is_Recent'] = (df_features['Year'] >= 2015).astype(int)

    # 3. Is_Major_Tech: 1 if Organisation belongs to the 10 defined major tech leaders
    major_tech_targets = [
        'yahoo', 'facebook', 'linkedin', 'twitter', 'google',
        'microsoft', 'amazon', 'apple', 'adobe', 'ebay'
    ]

    def identify_major_tech(org_name):
        # Case-insensitive substring match
        name_lower = str(org_name).lower()
        return int(any(tech in name_lower for tech in major_tech_targets))

    df_features['Is_Major_Tech'] = df_features['Organisation'].apply(identify_major_tech)

    # 4. Attack_Category: Group granular Method into 6 standardized categories
    def categorize_attack_method(method_text):
        m = str(method_text).lower()
        if 'hacked' in m or 'hack' in m:
            return 'Hacked'
        elif 'accidentally published' in m:
            return 'Accidentally Published'
        elif 'poor security' in m:
            return 'Poor Security'
        elif 'inside job' in m:
            return 'Inside Job'
        elif 'lost' in m or 'stolen' in m:
            return 'Lost Device'
        else:
            return 'Other'

    df_features['Attack_Category'] = df_features['Method'].apply(categorize_attack_method)

    # Print Feature Engineering Summary
    print("="*60)
    print("FEATURE ENGINEERING COMPLETED")
    print("="*60)
    print(f"Features in Dataset: {df_features.columns.tolist()}\n")
    print("Is_Recent Distribution:")
    print(df_features['Is_Recent'].value_counts().rename({1: 'Recent (>=2015)', 0: 'Historical (<2015)'}))
    print("\nIs_Major_Tech Distribution:")
    print(df_features['Is_Major_Tech'].value_counts().rename({1: 'Major Tech Giant', 0: 'Other Organization'}))
    print("\nAttack_Category Distribution:")
    print(df_features['Attack_Category'].value_counts())
    print("\nLog_Records Summary Statistics:")
    print(df_features['Log_Records'].describe().round(3))

except Exception as err:
    print(f" Error in Step 3: {err}")


---
## STEP 4 — EXPLORATORY DATA ANALYSIS (EDA)
In this section, we answer 8 core analytical questions with code and **11 professional cybersecurity intelligence visualizations**:
1. *How many data breaches occurred each year?*
2. *Which industry sectors are most heavily targeted?*
3. *Which attack vectors are most prevalent?*
4. *Which organisations suffered the largest record losses?*
5. *How has breach severity evolved across time?*
6. *Which decade witnessed the largest breach volume?*
7. *What does the record loss frequency distribution look like?*
8. *How do different attack vectors compare in terms of record destruction?*
9. *What is the average record loss per incident across sectors?*
10. *What are the correlations between numeric and ordinal security features?*
11. *How are breach severity tiers distributed across historical decades?*


In [5]:
# ==========================================
# STEP 4: EXPLORATORY DATA ANALYSIS (EDA)
# 11 PROFESSIONAL CYBERSECURITY VISUALIZATIONS
# ==========================================

# 1. Breaches Per Year
plt.figure(figsize=(12, 5))
yearly_counts = df_features.groupby('Year').size()
ax1 = sns.barplot(x=yearly_counts.index, y=yearly_counts.values, palette='viridis')
plt.title('Visualization 1: Global Data Breaches Frequency per Year (1984 - 2024)', weight='bold', pad=15)
plt.xlabel('Incident Year', weight='bold')
plt.ylabel('Number of Breaches Reported', weight='bold')
plt.xticks(rotation=45)
for p in ax1.patches:
    if p.get_height() > 0:
        ax1.annotate(f"{int(p.get_height())}", (p.get_x() + p.get_width() / 2., p.get_height()),
                     ha='center', va='bottom', fontsize=8, xytext=(0, 2), textcoords='offset points')
plt.tight_layout()
plt.show()

# 2. Top Targeted Industry Sectors
plt.figure(figsize=(10, 5))
top_10_sectors = df_features['Sector'].value_counts().head(10)
ax2 = sns.barplot(x=top_10_sectors.values, y=top_10_sectors.index, palette='mako')
plt.title('Visualization 2: Top 10 Most Targeted Industry Sectors', weight='bold', pad=15)
plt.xlabel('Breach Incident Count', weight='bold')
plt.ylabel('Sector', weight='bold')
for p in ax2.patches:
    ax2.annotate(f"{int(p.get_width())}", (p.get_width(), p.get_y() + p.get_height()/2.),
                 va='center', ha='left', fontsize=9, xytext=(5, 0), textcoords='offset points')
plt.tight_layout()
plt.show()

# 3. Common Attack Methods
plt.figure(figsize=(9, 5))
attack_dist = df_features['Attack_Category'].value_counts()
ax3 = sns.barplot(x=attack_dist.values, y=attack_dist.index, palette='rocket')
plt.title('Visualization 3: Distribution of Common Attack Vectors / Categories', weight='bold', pad=15)
plt.xlabel('Breach Incident Count', weight='bold')
plt.ylabel('Attack Vector Category', weight='bold')
for p in ax3.patches:
    pct = (p.get_width() / len(df_features)) * 100
    ax3.annotate(f"{int(p.get_width())} ({pct:.1f}%)", (p.get_width(), p.get_y() + p.get_height()/2.),
                 va='center', ha='left', fontsize=9, xytext=(5, 0), textcoords='offset points')
plt.tight_layout()
plt.show()

# 4. Top 10 Organisations by Total Records Lost
plt.figure(figsize=(11, 5))
top_orgs = df_features.groupby('Organisation')['Records Lost'].sum().sort_values(ascending=False).head(10)
top_orgs_billions = top_orgs / 1e9
ax4 = sns.barplot(x=top_orgs_billions.values, y=top_orgs.index, palette='flare')
plt.title('Visualization 4: Top 10 Organisations by Cumulative Records Lost (Billions)', weight='bold', pad=15)
plt.xlabel('Records Lost (in Billions)', weight='bold')
plt.ylabel('Organisation', weight='bold')
for p in ax4.patches:
    ax4.annotate(f"{p.get_width():.2f}B", (p.get_width(), p.get_y() + p.get_height()/2.),
                 va='center', ha='left', fontsize=9, xytext=(5, 0), textcoords='offset points')
plt.tight_layout()
plt.show()

# 5. Breach Severity Distribution Over Time (Stacked Bar)
plt.figure(figsize=(12, 5))
sev_time = pd.crosstab(df_features['Year'], df_features['Severity'])[['Low', 'Medium', 'High', 'Critical']]
ax5 = sev_time.plot(kind='bar', stacked=True, figsize=(12, 5), color=['#2ca02c', '#1f77b4', '#ff7f0e', '#d62728'], width=0.85)
plt.title('Visualization 5: Breach Severity Tier Composition Over Time', weight='bold', pad=15)
plt.xlabel('Year', weight='bold')
plt.ylabel('Breach Count', weight='bold')
plt.xticks(rotation=45)
plt.legend(title='Severity Tier', frameon=True)
plt.tight_layout()
plt.show()

# 6. Breaches Distribution by Decade
plt.figure(figsize=(8, 4))
decade_counts = df_features['Decade'].value_counts()[['1980s', '1990s', '2000s', '2010s', '2020s']]
ax6 = sns.barplot(x=decade_counts.index, y=decade_counts.values, palette='cubehelix')
plt.title('Visualization 6: Historical Breach Count by Decade', weight='bold', pad=15)
plt.xlabel('Decade', weight='bold')
plt.ylabel('Total Incidents', weight='bold')
for p in ax6.patches:
    pct = (p.get_height() / len(df_features)) * 100
    ax6.annotate(f"{int(p.get_height())} ({pct:.1f}%)", (p.get_x() + p.get_width()/2., p.get_height()),
                 ha='center', va='bottom', fontsize=9, xytext=(0, 3), textcoords='offset points')
plt.tight_layout()
plt.show()

# 7. Distribution of Records Lost (Histogram with KDE)
plt.figure(figsize=(10, 5))
sns.histplot(df_features['Log_Records'], kde=True, bins=25, color='#1f77b4', edgecolor='black')
plt.title('Visualization 7: Distribution of Compromised Records (Log10 Scale)', weight='bold', pad=15)
plt.xlabel('Log10(Records Lost)', weight='bold')
plt.ylabel('Frequency of Incidents', weight='bold')
plt.axvline(6, color='#2ca02c', linestyle='--', label='1M Threshold (Medium)')
plt.axvline(7, color='#ff7f0e', linestyle='--', label='10M Threshold (High)')
plt.axvline(8, color='#d62728', linestyle='--', label='100M Threshold (Critical)')
plt.legend(frameon=True)
plt.tight_layout()
plt.show()

# 8. Records Lost by Attack Category (Boxplot on Log Scale)
plt.figure(figsize=(10, 5))
sns.boxplot(x='Attack_Category', y='Log_Records', data=df_features, palette='Set2')
plt.title('Visualization 8: Breach Magnitude by Attack Category (Log Scale)', weight='bold', pad=15)
plt.xlabel('Attack Category', weight='bold')
plt.ylabel('Log10(Records Lost)', weight='bold')
plt.tight_layout()
plt.show()

# 9. Average Records Lost per Breach by Sector
plt.figure(figsize=(10, 5))
sector_avg = df_features.groupby('Sector')['Records Lost'].mean().sort_values(ascending=False).head(10) / 1e6
ax9 = sns.barplot(x=sector_avg.values, y=sector_avg.index, palette='crest')
plt.title('Visualization 9: Top 10 Sectors by Average Records Lost per Breach (Millions)', weight='bold', pad=15)
plt.xlabel('Average Records Lost (Millions)', weight='bold')
plt.ylabel('Industry Sector', weight='bold')
for p in ax9.patches:
    ax9.annotate(f"{p.get_width():.1f}M", (p.get_width(), p.get_y() + p.get_height()/2.),
                 va='center', ha='left', fontsize=9, xytext=(5, 0), textcoords='offset points')
plt.tight_layout()
plt.show()

# 10. Correlation Matrix Heatmap
plt.figure(figsize=(8, 6))
corr_subset = df_features[['Year', 'Records Lost', 'Log_Records', 'Is_Recent', 'Is_Major_Tech']].copy()
corr_subset['Severity_Ordinal'] = df_features['Severity'].map({'Low': 1, 'Medium': 2, 'High': 3, 'Critical': 4})
sns.heatmap(corr_subset.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Visualization 10: Feature Correlation Matrix Heatmap', weight='bold', pad=15)
plt.tight_layout()
plt.show()

# 11. Severity Level Breakdown by Decade
plt.figure(figsize=(10, 5))
decade_sev = pd.crosstab(df_features['Decade'], df_features['Severity'])[['Low', 'Medium', 'High', 'Critical']]
decade_sev.plot(kind='bar', figsize=(10, 5), color=['#2ca02c', '#1f77b4', '#ff7f0e', '#d62728'], width=0.8)
plt.title('Visualization 11: Severity Tier Distribution Across Historical Decades', weight='bold', pad=15)
plt.xlabel('Decade', weight='bold')
plt.ylabel('Incident Frequency', weight='bold')
plt.xticks(rotation=0)
plt.legend(title='Severity', frameon=True)
plt.tight_layout()
plt.show()


---
## STEP 5 — STATISTICAL ANALYSIS
Here, we perform rigorous statistical synthesis on global breach parameters:
- **Global Record Casualties:** Cumulative total records leaked worldwide.
- **Incident Central Tendencies:** Mean and median records exposed per breach.
- **Sector Vulnerabilities:** Breach count distributions by industry sector.
- **Attack Category Volume:** Incident frequencies per attack vector.
- **Severity Record Destruction:** Average records compromised per severity tier.
- **Temporal Growth Dynamics:** Year-over-year percentage expansion rates.
- **Top 10 Victim Entities:** Cumulative record exposures across target entities.
- **Top 5 Most Destructive Attack Vectors:** Attack categories ranked by average impact.


In [6]:
# ==========================================
# STEP 5: STATISTICAL ANALYSIS
# ==========================================

try:
    # 1. Total records lost globally
    total_records = df_features['Records Lost'].sum()

    # 2. Average records lost per breach
    avg_records = df_features['Records Lost'].mean()
    median_records = df_features['Records Lost'].median()

    # 3. Breach count by sector
    sector_counts = df_features['Sector'].value_counts()

    # 4. Breach count by attack method
    method_counts = df_features['Attack_Category'].value_counts()

    # 5. Average records lost by severity level
    severity_order = ['Low', 'Medium', 'High', 'Critical']
    avg_records_severity = df_features.groupby('Severity')['Records Lost'].mean().reindex(severity_order)

    # 6. Year over year breach growth rate
    yoy_df = df_features.groupby('Year').agg(
        breaches=('Organisation', 'count'),
        records=('Records Lost', 'sum')
    ).reset_index().sort_values('Year')
    yoy_df['Previous_Year_Breaches'] = yoy_df['breaches'].shift(1)
    yoy_df['YoY_Growth_Rate_Pct'] = (
        (yoy_df['breaches'] - yoy_df['Previous_Year_Breaches']) / yoy_df['Previous_Year_Breaches'] * 100
    ).round(2)

    # 7. Top 10 most breached organisations (by records lost)
    top10_organisations = df_features.groupby('Organisation').agg(
        total_records_lost=('Records Lost', 'sum'),
        incident_count=('Year', 'count'),
        sector=('Sector', 'first')
    ).sort_values(by='total_records_lost', ascending=False).head(10)

    # 8. Top 5 most dangerous attack methods by average records lost
    top5_attack_methods = df_features.groupby('Attack_Category').agg(
        avg_records_lost=('Records Lost', 'mean'),
        total_records_lost=('Records Lost', 'sum'),
        total_incidents=('Year', 'count')
    ).sort_values(by='avg_records_lost', ascending=False).head(5)

    # Format and Display Results
    print("="*75)
    print("GLOBAL CYBERSECURITY STATISTICAL DOSSIER")
    print("="*75)
    print(f"1. Total Records Lost Globally:        {total_records:>18,d} records")
    print(f"2. Average Records Lost per Incident:  {avg_records:>18,.0f} records")
    print(f"   Median Records Lost per Incident:   {median_records:>18,.0f} records")

    print("\n3. Breach Count by Top 5 Targeted Sectors:")
    for s, c in sector_counts.head(5).items():
        print(f"   - {s:<20}: {c:>3} breaches ({(c/len(df_features)*100):.1f}%)")

    print("\n4. Breach Count by Attack Method:")
    for m, c in method_counts.items():
        print(f"   - {m:<24}: {c:>3} breaches ({(c/len(df_features)*100):.1f}%)")

    print("\n5. Average Records Lost by Severity Level:")
    for s, val in avg_records_severity.items():
        print(f"   - {s:<10}: {val:>15,.0f} average records")

    print("\n6. Year-Over-Year Breach Growth Rate (Sample 2013-2020):")
    recent_yoy = yoy_df[yoy_df['Year'].between(2013, 2020)][['Year', 'breaches', 'Previous_Year_Breaches', 'YoY_Growth_Rate_Pct']]
    print(recent_yoy.to_string(index=False))

    print("\n7. Top 10 Most Breached Organisations:")
    for i, (org, row) in enumerate(top10_organisations.iterrows(), 1):
        print(f"   {i:>2}. {org:<34} ({row['sector']:<10}): {row['total_records_lost']:>14,d} records ({row['incident_count']} incidents)")

    print("\n8. Top 5 Most Dangerous Attack Methods (by Average Impact):")
    for i, (meth, row) in enumerate(top5_attack_methods.iterrows(), 1):
        print(f"   {i:>2}. {meth:<24}: {row['avg_records_lost']:>14,.0f} avg records ({row['total_incidents']} incidents)")

except Exception as err:
    print(f" Error in Step 5: {err}")


---
## STEP 6 — MACHINE LEARNING: BREACH SEVERITY PREDICTION
To automate incident triage and predictive threat rating, we deploy multi-class machine learning models to forecast **Severity**:
- **Target:** `Severity` encoded (`Critical`, `High`, `Medium`, `Low`).
- **Features:** `Year`, `Log_Records`, `Is_Recent`, `Is_Major_Tech`, `Attack_Category_Encoded`, `Sector_Encoded`, `Method_Encoded`.
- **Validation Protocol:** 80% Train, 20% Test with **Stratified Sampling** to guarantee class preservation.
- **Normalization:** `StandardScaler` feature standardization.
- **Algorithms:**
  1. **Random Forest Classifier** (`n_estimators=100`, `random_state=42`)
  2. **XGBoost Classifier** (`n_estimators=100`, `random_state=42`, `eval_metric='mlogloss'`)
  3. **Logistic Regression** (`max_iter=1000`, `random_state=42`)
- **Evaluation:** Accuracy, Precision, Recall, F1-Score, Confusion Matrix Heatmaps, and Feature Importance charts.


In [7]:
# ==========================================
# STEP 6: MACHINE LEARNING MODELING
# ==========================================

try:
    df_ml = df_features.copy()

    # Categorical Label Encoders
    le_severity = LabelEncoder()
    df_ml['Severity_Encoded'] = le_severity.fit_transform(df_ml['Severity'])

    le_attack = LabelEncoder()
    df_ml['Attack_Category_Encoded'] = le_attack.fit_transform(df_ml['Attack_Category'])

    le_sector = LabelEncoder()
    df_ml['Sector_Encoded'] = le_sector.fit_transform(df_ml['Sector'])

    le_method = LabelEncoder()
    df_ml['Method_Encoded'] = le_method.fit_transform(df_ml['Method'])

    # Assemble Feature Matrix X and Target Vector y
    ml_feature_names = [
        'Year', 'Log_Records', 'Is_Recent', 'Is_Major_Tech',
        'Attack_Category_Encoded', 'Sector_Encoded', 'Method_Encoded'
    ]
    X = df_ml[ml_feature_names]
    y = df_ml['Severity_Encoded']

    # 80% Train, 20% Test split with stratified class distribution
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y
    )

    # Standardize feature scales
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    severity_class_labels = le_severity.classes_

    # -------------------------------------------------------------
    # MODEL 1 — RANDOM FOREST CLASSIFIER
    # -------------------------------------------------------------
    print("="*60)
    print("TRAINING MODEL 1: RANDOM FOREST CLASSIFIER")
    print("="*60)
    rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
    rf_classifier.fit(X_train_scaled, y_train)
    y_pred_rf = rf_classifier.predict(X_test_scaled)

    acc_rf = accuracy_score(y_test, y_pred_rf)
    prec_rf = precision_score(y_test, y_pred_rf, average='weighted', zero_division=0)
    rec_rf = recall_score(y_test, y_pred_rf, average='weighted', zero_division=0)
    f1_rf = f1_score(y_test, y_pred_rf, average='weighted', zero_division=0)

    print(f"Random Forest Accuracy: {acc_rf:.4f} ({acc_rf*100:.2f}%)")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred_rf, target_names=severity_class_labels))

    # Confusion Matrix Heatmap - RF
    plt.figure(figsize=(6, 5))
    cm_rf = confusion_matrix(y_test, y_pred_rf)
    sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Blues',
                xticklabels=severity_class_labels, yticklabels=severity_class_labels)
    plt.title(f'Random Forest Confusion Matrix (Acc: {acc_rf*100:.1f}%)', weight='bold')
    plt.xlabel('Predicted Severity Tier')
    plt.ylabel('Ground Truth Severity Tier')
    plt.tight_layout()
    plt.show()

    # Feature Importance - RF
    plt.figure(figsize=(8, 4))
    rf_feat_imp = pd.Series(rf_classifier.feature_importances_, index=ml_feature_names).sort_values(ascending=True)
    rf_feat_imp.plot(kind='barh', color='#1f77b4')
    plt.title('Random Forest Feature Importance Weights', weight='bold')
    plt.xlabel('Gini Importance')
    plt.tight_layout()
    plt.show()

    # -------------------------------------------------------------
    # MODEL 2 — XGBOOST CLASSIFIER
    # -------------------------------------------------------------
    print("\n" + "="*60)
    print("TRAINING MODEL 2: XGBOOST CLASSIFIER")
    print("="*60)
    xgb_classifier = XGBClassifier(n_estimators=100, random_state=42, eval_metric='mlogloss')
    xgb_classifier.fit(X_train_scaled, y_train)
    y_pred_xgb = xgb_classifier.predict(X_test_scaled)

    acc_xgb = accuracy_score(y_test, y_pred_xgb)
    prec_xgb = precision_score(y_test, y_pred_xgb, average='weighted', zero_division=0)
    rec_xgb = recall_score(y_test, y_pred_xgb, average='weighted', zero_division=0)
    f1_xgb = f1_score(y_test, y_pred_xgb, average='weighted', zero_division=0)

    print(f"XGBoost Accuracy: {acc_xgb:.4f} ({acc_xgb*100:.2f}%)")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred_xgb, target_names=severity_class_labels))

    # Confusion Matrix Heatmap - XGB
    plt.figure(figsize=(6, 5))
    cm_xgb = confusion_matrix(y_test, y_pred_xgb)
    sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Greens',
                xticklabels=severity_class_labels, yticklabels=severity_class_labels)
    plt.title(f'XGBoost Confusion Matrix (Acc: {acc_xgb*100:.1f}%)', weight='bold')
    plt.xlabel('Predicted Severity Tier')
    plt.ylabel('Ground Truth Severity Tier')
    plt.tight_layout()
    plt.show()

    # Feature Importance - XGB
    plt.figure(figsize=(8, 4))
    xgb_feat_imp = pd.Series(xgb_classifier.feature_importances_, index=ml_feature_names).sort_values(ascending=True)
    xgb_feat_imp.plot(kind='barh', color='#2ca02c')
    plt.title('XGBoost Feature Importance Weights', weight='bold')
    plt.xlabel('F-Score / Relative Weight')
    plt.tight_layout()
    plt.show()

    # -------------------------------------------------------------
    # MODEL 3 — LOGISTIC REGRESSION
    # -------------------------------------------------------------
    print("\n" + "="*60)
    print("TRAINING MODEL 3: LOGISTIC REGRESSION")
    print("="*60)
    lr_classifier = LogisticRegression(max_iter=1000, random_state=42)
    lr_classifier.fit(X_train_scaled, y_train)
    y_pred_lr = lr_classifier.predict(X_test_scaled)

    acc_lr = accuracy_score(y_test, y_pred_lr)
    prec_lr = precision_score(y_test, y_pred_lr, average='weighted', zero_division=0)
    rec_lr = recall_score(y_test, y_pred_lr, average='weighted', zero_division=0)
    f1_lr = f1_score(y_test, y_pred_lr, average='weighted', zero_division=0)

    print(f"Logistic Regression Accuracy: {acc_lr:.4f} ({acc_lr*100:.2f}%)")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred_lr, target_names=severity_class_labels))

    # Confusion Matrix Heatmap - LR
    plt.figure(figsize=(6, 5))
    cm_lr = confusion_matrix(y_test, y_pred_lr)
    sns.heatmap(cm_lr, annot=True, fmt='d', cmap='Oranges',
                xticklabels=severity_class_labels, yticklabels=severity_class_labels)
    plt.title(f'Logistic Regression Confusion Matrix (Acc: {acc_lr*100:.1f}%)', weight='bold')
    plt.xlabel('Predicted Severity Tier')
    plt.ylabel('Ground Truth Severity Tier')
    plt.tight_layout()
    plt.show()

    # Model Performance Summary Comparison Table
    print("\n" + "="*60)
    print("MODEL COMPARISON BENCHMARK TABLE")
    print("="*60)
    model_benchmarks = pd.DataFrame({
        'Model Name': ['Random Forest Classifier', 'XGBoost Classifier', 'Logistic Regression'],
        'Accuracy': [acc_rf, acc_xgb, acc_lr],
        'Precision': [prec_rf, prec_xgb, prec_lr],
        'Recall': [rec_rf, rec_xgb, rec_lr],
        'F1-Score': [f1_rf, f1_xgb, f1_lr]
    })
    print(model_benchmarks.to_string(index=False, formatters={
        'Accuracy': '{:.2%}'.format,
        'Precision': '{:.2%}'.format,
        'Recall': '{:.2%}'.format,
        'F1-Score': '{:.2%}'.format
    }))

except Exception as err:
    print(f" Error in Step 6: {err}")


---
## STEP 7 — THREAT INTELLIGENCE INSIGHTS
Threat intelligence contextualizes retrospective breach statistics into forward-looking cyber defense priorities:
1. **Most Dangerous Year:** Identified across both record destruction and sheer incident volume.
2. **Fastest Growing Threat Vector:** Measured by comparing post-2015 vector representation against historical baselines.
3. **Conditional Mega-Breach Probability:** $P(\text{Severity} = \text{'Critical'} \mid \text{Sector})$.
4. **Composite Sector Vulnerability Index:** Combining breach frequency, total blast radius, and critical probability.
5. **Linear Breach Frequency Trajectory:** Quantifying whether breach frequency is accelerating or attenuating.


In [8]:
# ==========================================
# STEP 7: THREAT INTELLIGENCE INSIGHTS
# ==========================================

try:
    # 1. Most Dangerous Year
    year_stats = df_features.groupby('Year').agg(
        records_total=('Records Lost', 'sum'),
        breach_count=('Organisation', 'count')
    ).reset_index()

    deadliest_records_year = year_stats.loc[year_stats['records_total'].idxmax()]
    deadliest_count_year = year_stats.loc[year_stats['breach_count'].idxmax()]

    # 2. Fastest Growing Attack Method
    # Compare post-2015 distribution against pre-2015
    period_crosstab = pd.crosstab(df_features['Attack_Category'], df_features['Year'] >= 2015, normalize='index')
    period_growth = (period_crosstab[True] - period_crosstab[False]).sort_values(ascending=False)
    fastest_vector = period_growth.index[0]
    fastest_vector_growth = period_growth.iloc[0] * 100

    # 3. Probability of a breach being Critical severity by sector
    critical_prob_by_sector = df_features.groupby('Sector').apply(
        lambda group: (group['Severity'] == 'Critical').mean(), include_groups=False
    ).sort_values(ascending=False)

    # 4. Identify Most Vulnerable Sectors (Composite Cyber Risk Index)
    sector_risk = df_features.groupby('Sector').agg(
        total_breaches=('Year', 'count'),
        total_records=('Records Lost', 'sum'),
        crit_prob=('Severity', lambda s: (s == 'Critical').mean())
    ).reset_index()

    # Calculate normalized composite vulnerability score (0 to 100)
    sector_risk['Vulnerability_Score'] = (
        (sector_risk['total_breaches'] / sector_risk['total_breaches'].max()) * 0.35 +
        (sector_risk['total_records'] / sector_risk['total_records'].max()) * 0.35 +
        (sector_risk['crit_prob'] / 1.0) * 0.30
    ) * 100
    sector_risk = sector_risk.sort_values(by='Vulnerability_Score', ascending=False)

    # 5. Breach Frequency Trend (Regression Analysis)
    modern_trend_data = df_features[df_features['Year'] >= 2004].groupby('Year').size().reset_index(name='count')
    slope, intercept, r_val, p_val, std_err = stats.linregress(modern_trend_data['Year'], modern_trend_data['count'])
    trend_state = "INCREASING (Upward expansion)" if slope > 0 else "DECREASING"

    # Display Threat Intelligence Report
    print("="*75)
    print("STRATEGIC CYBER THREAT INTELLIGENCE REPORT")
    print("="*75)
    print(f" 1. Most Dangerous Year (By Blast Radius): {int(deadliest_records_year['Year'])} ({deadliest_records_year['records_total']:,.0f} records)")
    print(f"    Most Dangerous Year (By Incident Count): {int(deadliest_count_year['Year'])} ({int(deadliest_count_year['breach_count'])} reported breaches)")
    print(f"\n 2. Fastest Growing Attack Vector:       {fastest_vector} (+{fastest_vector_growth:.1f}% post-2015 surge)")
    print(f"\n 3. Top Sectors by Critical Severity Risk [P(Critical | Sector)]:")
    for sec, prob in critical_prob_by_sector.head(5).items():
        print(f"    - {sec:<20}: {prob*100:>5.1f}% chance of exceeding 100M records")
    print(f"\n 4. Top 3 Most Vulnerable Industry Sectors:")
    for idx, row in sector_risk.head(3).iterrows():
        print(f"    - {row['Sector']:<20}: Risk Score = {row['Vulnerability_Score']:>5.1f}/100 ({row['total_breaches']} breaches, {row['total_records']:,.0f} records)")
    print(f"\n 5. Global Breach Frequency Trajectory: {trend_state}")
    print(f"    Slope: +{slope:.3f} incidents/year | Correlation R^2: {r_val**2:.3f}")

except Exception as err:
    print(f" Error in Step 7: {err}")


---
## STEP 8 — LOAD INTO MYSQL DATABASE & ADVANCED SQL QUERIES
Enterprise security architectures store curated breach telemetry in relational database warehouses to support audits, BI dashboards, and ad-hoc investigations.

In this step, we:
1. Connect to **MySQL Server** using the specified authentication configuration.
2. Initialize database **`cybersecurity_db`** if not already created.
3. Define the relational schema and create table **`data_breaches`**.
4. Perform bulk insertion of all cleaned records.
5. Execute **5 advanced analytical SQL queries**:
   - **Query 1:** Sector aggregations with `GROUP BY` and `ORDER BY`.
   - **Query 2:** Attack method danger ratings using `GROUP BY` and `HAVING`.
   - **Query 3:** Longitudinal severity classification by decade with `CASE WHEN`.
   - **Query 4:** Year-over-year breach velocity using Window Function `LAG()`.
   - **Query 5:** Top 10 most breached organisations using **Common Table Expressions (CTE)** and `LIMIT`.


In [9]:
# ==========================================
# STEP 8: LOAD INTO MYSQL & ADVANCED QUERIES
# ==========================================

import sys
sys.path.append(r'C:\Users\HP\AppData\Roaming\Python\Python313\site-packages')
import mysql.connector

try:
    # 1. Establish initial connection to verify/create database
    admin_conn = mysql.connector.connect(
        host='localhost',
        user='root',
        password='Saad2025',
        auth_plugin='mysql_native_password'
    )
    admin_cursor = admin_conn.cursor()
    admin_cursor.execute("CREATE DATABASE IF NOT EXISTS cybersecurity_db;")
    admin_cursor.close()
    admin_conn.close()

    # 2. Connect to cybersecurity_db
    conn = mysql.connector.connect(
        host='localhost',
        user='root',
        password='Saad2025',
        database='cybersecurity_db',
        auth_plugin='mysql_native_password'
    )
    cursor = conn.cursor()
    print(" Successfully connected to MySQL database: `cybersecurity_db`")

    # 3. Drop existing table and create fresh schema
    cursor.execute("DROP TABLE IF EXISTS data_breaches;")
    create_table_ddl = """
    CREATE TABLE data_breaches (
        id INT AUTO_INCREMENT PRIMARY KEY,
        organisation VARCHAR(255) NOT NULL,
        year INT NOT NULL,
        records_lost BIGINT NOT NULL,
        sector VARCHAR(100),
        method VARCHAR(255),
        severity VARCHAR(50),
        decade VARCHAR(20),
        log_records DOUBLE,
        is_recent INT,
        is_major_tech INT,
        attack_category VARCHAR(100)
    );
    """
    cursor.execute(create_table_ddl)
    print(" Table `data_breaches` created successfully.")

    # 4. Prepare batch insert statement
    insert_sql = """
    INSERT INTO data_breaches 
    (organisation, year, records_lost, sector, method, severity, decade, log_records, is_recent, is_major_tech, attack_category)
    VALUES (%s, %s, %s, %s, %s, %s, %s, %s, %s, %s, %s);
    """

    batch_data = [
        (
            row['Organisation'], int(row['Year']), int(row['Records Lost']),
            row['Sector'], row['Method'], row['Severity'], row['Decade'],
            float(row['Log_Records']), int(row['Is_Recent']), int(row['Is_Major_Tech']),
            row['Attack_Category']
        )
        for _, row in df_features.iterrows()
    ]

    cursor.executemany(insert_sql, batch_data)
    conn.commit()

    cursor.execute("SELECT COUNT(*) FROM data_breaches;")
    inserted_count = cursor.fetchone()[0]
    print(f" Data Ingestion Success: {inserted_count} incident records inserted into MySQL.")

    # -------------------------------------------------------------
    # 5 ADVANCED SQL QUERIES EXECUTION
    # -------------------------------------------------------------

    # Query 1: Total records lost and breach count by sector
    q1 = """
    SELECT 
        sector,
        COUNT(*) AS breach_count,
        SUM(records_lost) AS total_records_lost,
        ROUND(AVG(records_lost), 0) AS avg_records_lost
    FROM data_breaches
    GROUP BY sector
    ORDER BY total_records_lost DESC;
    """
    print("\n" + "="*70)
    print("SQL QUERY 1: Total Records Lost and Breach Count by Sector")
    print("="*70)
    cursor.execute(q1)
    for r in cursor.fetchall()[:7]:
        print(f"Sector: {r[0]:<15} | Breaches: {r[1]:<3} | Total Records: {r[2]:>14,f} | Avg Records: {r[3]:>12,f}")

    # Query 2: Most dangerous attack methods by average records lost
    q2 = """
    SELECT 
        attack_category,
        COUNT(*) AS incident_count,
        ROUND(AVG(records_lost), 0) AS avg_records_lost,
        MAX(records_lost) AS max_records_lost
    FROM data_breaches
    GROUP BY attack_category
    HAVING COUNT(*) >= 5
    ORDER BY avg_records_lost DESC;
    """
    print("\n" + "="*70)
    print("SQL QUERY 2: Most Dangerous Attack Methods by Average Records Lost")
    print("="*70)
    cursor.execute(q2)
    for r in cursor.fetchall():
        print(f"Attack: {r[0]:<22} | Incidents: {r[1]:<3} | Avg Records: {r[2]:>12,f} | Max Records: {r[3]:>14,f}")

    # Query 3: Breach severity classification by decade
    q3 = """
    SELECT 
        decade,
        COUNT(*) AS total_breaches,
        SUM(CASE WHEN severity = 'Critical' THEN 1 ELSE 0 END) AS critical_count,
        SUM(CASE WHEN severity = 'High' THEN 1 ELSE 0 END) AS high_count,
        SUM(CASE WHEN severity = 'Medium' THEN 1 ELSE 0 END) AS medium_count,
        SUM(CASE WHEN severity = 'Low' THEN 1 ELSE 0 END) AS low_count,
        ROUND(100.0 * SUM(CASE WHEN severity = 'Critical' THEN 1 ELSE 0 END) / COUNT(*), 2) AS critical_pct
    FROM data_breaches
    GROUP BY decade
    ORDER BY decade ASC;
    """
    print("\n" + "="*70)
    print("SQL QUERY 3: Breach Severity Classification by Decade")
    print("="*70)
    cursor.execute(q3)
    for r in cursor.fetchall():
        print(f"Decade: {r[0]:<6} | Total: {r[1]:<3} | Crit: {r[2]:<2} | High: {r[3]:<2} | Med: {r[4]:<2} | Low: {r[5]:<2} | Crit %: {r[6]}%")

    # Query 4: Year over year breach growth using LAG()
    q4 = """
    WITH yearly_stats AS (
        SELECT 
            year,
            COUNT(*) AS breach_count,
            SUM(records_lost) AS total_records
        FROM data_breaches
        GROUP BY year
    )
    SELECT 
        year,
        breach_count,
        LAG(breach_count, 1) OVER (ORDER BY year) AS prev_year_breaches,
        breach_count - LAG(breach_count, 1) OVER (ORDER BY year) AS yoy_breach_change,
        ROUND(100.0 * (breach_count - LAG(breach_count, 1) OVER (ORDER BY year)) / NULLIF(LAG(breach_count, 1) OVER (ORDER BY year), 0), 2) AS yoy_breach_growth_pct,
        total_records
    FROM yearly_stats
    ORDER BY year ASC;
    """
    print("\n" + "="*70)
    print("SQL QUERY 4: Year-over-Year Breach Growth Using Window Function LAG()")
    print("="*70)
    cursor.execute(q4)
    rows_q4 = cursor.fetchall()
    for r in [row for row in rows_q4 if 2012 <= row[0] <= 2020]:
        prev = r[2] if r[2] is not None else 0
        chg = r[3] if r[3] is not None else 0
        pct = f"{r[4]:.1f}%" if r[4] is not None else "N/A"
        print(f"Year {r[0]}: Breaches={r[1]:<2} | Prev={prev:<2} | Change={chg:<+3} | YoY Growth={pct}")

    # Query 5: Top 10 most breached organisations using CTE and ORDER BY LIMIT
    q5 = """
    WITH org_breach_summary AS (
        SELECT 
            organisation,
            sector,
            COUNT(*) AS total_breaches,
            SUM(records_lost) AS total_records_compromised,
            MAX(records_lost) AS single_largest_breach
        FROM data_breaches
        GROUP BY organisation, sector
    )
    SELECT 
        organisation,
        sector,
        total_breaches,
        total_records_compromised,
        single_largest_breach,
        DENSE_RANK() OVER (ORDER BY total_records_compromised DESC) AS breach_rank
    FROM org_breach_summary
    ORDER BY total_records_compromised DESC
    LIMIT 10;
    """
    print("\n" + "="*70)
    print("SQL QUERY 5: Top 10 Most Breached Organisations Using CTE & Dense Rank")
    print("="*70)
    cursor.execute(q5)
    for r in cursor.fetchall():
        print(f"Rank {r[5]:<2}: {r[0]:<28} ({r[1]:<10}) | Incidents: {r[2]} | Total Records: {r[3]:>14,f}")

    cursor.close()
    conn.close()

except Exception as err:
    print(f" MySQL Processing Error: {err}")


---
## STEP 9 — SAVE ALL ARTIFACTS & AUDIT TRAIL CONFIRMATION
To ensure seamless collaboration, reproducibility, and external dashboard integration (e.g., Tableau), all generated intelligence artifacts are persisted in `C:\Users\HP` and the project repository:
1. `cybersecurity_analysis.ipynb`: The complete executable Jupyter notebook.
2. `breaches_cleaned.csv`: Cleaned dataset with standard schemas.
3. `breaches_features.csv`: Engineered feature set ready for ML and BI modeling.
4. `cybersecurity_queries.sql`: The enterprise SQL script containing table schemas and analytical queries.


In [10]:
# ==========================================
# STEP 9: SAVE ALL FILES & VERIFY AUDIT TRAIL
# ==========================================

import os

try:
    target_locations = [r'C:\Users\HP', r'C:\Users\HP\Cybersecurity']

    # 1. Save cleaned dataset
    for loc in target_locations:
        clean_file = os.path.join(loc, 'breaches_cleaned.csv')
        df_cleaned.to_csv(clean_file, index=False)
        
    # 2. Save feature engineered dataset
    for loc in target_locations:
        feat_file = os.path.join(loc, 'breaches_features.csv')
        df_features.to_csv(feat_file, index=False)

    print("="*80)
    print("ARTIFACT VERIFICATION & AUDIT CONFIRMATION")
    print("="*80)
    
    files_to_confirm = [
        r'C:\Users\HP\cybersecurity_analysis.ipynb',
        r'C:\Users\HPreaches_cleaned.csv',
        r'C:\Users\HPreaches_features.csv',
        r'C:\Users\HP\cybersecurity_queries.sql',
        r'C:\Users\HP\Cybersecurityreaches_cleaned.csv',
        r'C:\Users\HP\Cybersecurityreaches_features.csv',
        r'C:\Users\HP\Cybersecurity\cybersecurity_queries.sql'
    ]

    for fpath in files_to_confirm:
        if os.path.exists(fpath):
            file_size_kb = os.path.getsize(fpath) / 1024
            print(f" [CONFIRMED] {fpath:<55} ({file_size_kb:>7.1f} KB)")
        else:
            print(f" [PENDING]   {fpath:<55} (will be created/saved)")

    print("\n ALL CYBERSECURITY DATA SCIENCE PROJECT WORKFLOW STEPS ARE COMPLETE!")

except Exception as err:
    print(f" Error in Step 9: {err}")
